# Burn cost workflow demo: optional model editor

Use this notebook only when you want to edit fitted relativities.

Select any published candidate version for this SQL model, or leave the
package unset to open the latest. Publishing an editor session creates an
immutable `EDITOR_EDIT` child package.


In [ ]:
DATABASE_MODE = "remote"  # "local" or "remote"
RUNTIME_MODULE = "demo_sql_runtime"  # e.g. "project_runtime.database"; never put secrets here
EXPECTED_REMOTE_DATABASE = "PricingNotebookDemo"
ALLOW_REMOTE_WRITES = False

MODEL_NAME = "DEMO_BURN_COST"  # Set to None to select by label only.
MODEL_LABEL = "Burn cost workflow demo"
DEPLOYMENT_SLOT = "DEMO_BURN_COST_ONLY"
PACKAGE_VERSION = None  # Choose after reading the package list.
EDIT_REASON = ""

In [ ]:
from pathlib import Path

PROJECT_ROOT = next(
    candidate
    for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
    if (candidate / "pyproject.toml").is_file() and (candidate / "pricing_models").is_dir()
)

import sys

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from superglm.editor import EditorSession

from pricing_pipeline.notebook import (
    connect,
    list_model_versions,
    load_registered_model,
    load_model_version,
    publish_edits,
)

MODEL_DIR = PROJECT_ROOT / "pricing_models/burn_cost_demo"

## Connect, resolve the model label, and list versions


In [ ]:
pricing = connect(
    mode=DATABASE_MODE,
    runtime_module=RUNTIME_MODULE,
    local_root=MODEL_DIR / ".local",
    expected_remote_database=EXPECTED_REMOTE_DATABASE,
    allow_remote_writes=ALLOW_REMOTE_WRITES,
)
display(pricing.destination)

In [ ]:
model = load_registered_model(
    pricing,
    model_name=MODEL_NAME,
    model_label=MODEL_LABEL,
    deployment_slot=DEPLOYMENT_SLOT,
    source_root=MODEL_DIR,
)
versions = list_model_versions(pricing, model=model)
display(versions.rename(columns={"Model version": "Model version", "Parent": "Parent package"}))

## Select and open the exact candidate


In [ ]:
if versions.empty:
    raise LookupError("No candidate package versions were found.")
selected_package_version = (
    int(versions.iloc[0]["Package"]) if PACKAGE_VERSION is None else int(PACKAGE_VERSION)
)
if selected_package_version not in set(versions["Package"].astype(int)):
    raise ValueError("PACKAGE_VERSION is not in the displayed candidate list.")
reviewed = load_model_version(
    pricing,
    model=model,
    package_version=selected_package_version,
)
display(
    {
        "Package": reviewed.package_version,
        "Model version": reviewed.recipe_revision,
        "Kind": reviewed.technical.get("model_kind"),
        "Data as of": reviewed.technical.get("data_as_of_date"),
        "Parent package": reviewed.technical.get("parent_package_version"),
    }
)

## Open the live editor


In [ ]:
editor_session = EditorSession.from_model(
    reviewed.bundle.fitted_model,
    train_data=(
        reviewed.bundle.X,
        reviewed.bundle.y,
        reviewed.bundle.sample_weight,
        reviewed.bundle.offset,
    ),
    cv_report=reviewed.bundle.cv_report,
)
display(editor_session.widget())

## Preview the edited model without publishing


In [ ]:
edited_model = editor_session.to_model()
edited_model

## Publish the retained editor session


In [ ]:
if not EDIT_REASON.strip():
    raise ValueError("Describe the market or underwriting edit.")
edited = publish_edits(
    pricing,
    candidate=reviewed,
    editor_session=editor_session,
    reason=EDIT_REASON,
)
display(
    {
        "Kind": edited.model_kind,
        "Package": edited.package_version,
        "Parent package ID": edited.parent_rate_package_id,
        "State": edited.package_status,
        "Reused equivalent": edited.deduplicated,
    }
)